# Name That LLM: LoRA fine-tuning with honest validation

**Purpose.** Answer one question with graphs: *does fine-tuning actually learn model identity on prompts it has never seen, or is it memorising?*

**What this notebook trains.** A truncated decoder (Qwen3 4B / 8B / 14B, first 24-27 layers) with LoRA adapters, mean pooling and a linear head. fp16 base weights, fp32 LoRA and head. It works on 1 or 2 T4s (T4 has no bf16, so fp16 + GradScaler).

**How it stays honest**
1. The validation fold is split by `prompt_id` (GroupKFold), so no validation prompt is seen in training.
2. A TF-IDF logistic regression baseline and a shuffled-label control run on the same split.
3. A train-subset accuracy (eval mode) is plotted next to validation accuracy, so you can see memorisation directly.
4. Optional **unseen set**: the dataset you generate yourself, with prompts that are not in `train.csv`. It is checked for near-duplicates of train before being scored.
5. Group assignment (Hungarian) is scored with pseudo-groups from clustering, which is what the hidden test will give you, as well as with the true groups.

**Outputs you will see.** Training-dynamics figure, a does-it-work bar chart, confusion matrix and per-class recall, a Hungarian sweep, `results_summary.csv`, `history.json`, saved adapter and probabilities.

**Smoke mode.** If `train.csv` is not found, the notebook runs on synthetic data with a sklearn stand-in for the LLM, only to check that every cell and graph works. Smoke numbers mean nothing.

## Settings
Edit `CFG` only. Rough costs are estimates, not measurements: run the first 10 optimizer steps and read the printed ETA before committing to a long run.

| backbone | layers kept | fp16 size | GPUs used |
|---|---|---|---|
| `qwen3_4b` | 24 | ~6 GB | one T4 (the second stays idle unless `gpu_mode='shard'`) |
| `qwen3_8b` | 24 | ~12 GB | both T4s, layers split across them |
| `qwen3_14b` | 27 | ~19 GB | both T4s, layers split across them |

Both-GPU mode is naive model parallelism: it makes big models fit, it does not make them faster.

In [ ]:
import os, re, gc, glob, json, time, math, random, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, log_loss, confusion_matrix, adjusted_rand_score)
warnings.filterwarnings('ignore')
try:
    import torch
    HAS_TORCH = True
except Exception:
    torch = None; HAS_TORCH = False
N_GPUS = torch.cuda.device_count() if HAS_TORCH else 0

CFG = dict(
    backbone='qwen3_4b',        # 'qwen3_4b' | 'qwen3_8b' | 'qwen3_14b'
    layers_kept=None,            # override preset: integer layer count; None uses preset
    gpu_mode='auto',            # 'auto' | 'single' | 'shard'
    max_len=384, lora_r=16, lora_alpha=32, lora_dropout=0.05,
    lora_targets=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    epochs=3, batch_size=4, grad_accum=4, lr=1e-4, head_lr=1e-3, weight_decay=0.01,
    warmup_frac=0.06, label_smooth=0.05, eval_bs=16, log_every=5,
    n_folds=5, val_fold=0,
    time_budget_hours=3.0,      # training stops cleanly after this and keeps the best epoch
    extra_paths=[],             # optional extra labelled files, TRAIN ONLY, e.g. ['/kaggle/input/xxx/gen.csv']
    extra_cap_per_class=300, extra_cap_per_source=0.30, extra_sample_by_match_weight=False,
    max_train_rows_per_epoch=12000, logit_adjustment=False, seeds=[42, 43],
    unseen_path=None,           # YOUR generated held-out set: csv/parquet with text, model (+ optional prompt_id)
    run_test=True,
    stage_to_tmp=True,          # copy the checkpoint to local /tmp once, then load from there (helps if /kaggle/input reads are slow)
)
LABEL_ALIASES = {}              # e.g. {'gpt-5-5': 'gpt-5.5'} if your generated files use other spellings
SEED = 42
random.seed(SEED); np.random.seed(SEED)
if HAS_TORCH: torch.manual_seed(SEED)

PRESETS = {
    'qwen3_4b':  dict(model_type='qwen3', hidden=2560, layers=36, keep=24, size_gb=6),
    'qwen3_8b':  dict(model_type='qwen3', hidden=4096, layers=36, keep=24, size_gb=12),
    'qwen3_14b': dict(model_type='qwen3', hidden=5120, layers=40, keep=27, size_gb=19),
}
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
FIG = os.path.join(OUT, 'figs'); os.makedirs(FIG, exist_ok=True)

import shutil
from concurrent.futures import ThreadPoolExecutor
_t = time.time()
INPUT_FILES = [os.path.join(r, f) for r, _, fs in os.walk('/kaggle/input') for f in fs] if os.path.isdir('/kaggle/input') else []
ST_DIRS = {os.path.dirname(p) for p in INPUT_FILES if p.endswith('.safetensors')}
print(f'indexed {len(INPUT_FILES)} input files once in {time.time()-_t:.0f}s (no further recursive globs)')
def find(name):
    pref = f'/kaggle/input/competitions/name-that-llm/{name}'      # prefer the competition's file when other datasets also contain one
    return pref if pref in INPUT_FILES else next((p for p in INPUT_FILES if p.endswith('/' + name)), None)

SMOKE = find('train.csv') is None
if not SMOKE:
    assert N_GPUS >= 1, 'Turn on the GPU accelerator (T4 x2) for this notebook.'
if HAS_TORCH and N_GPUS:
    for i in range(N_GPUS):
        pr = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {pr.name}  {pr.total_memory/2**30:.1f} GiB')
print('SMOKE MODE (synthetic data, sklearn stand-in)' if SMOKE else 'REAL MODE', '| GPUs:', N_GPUS)
if SMOKE:
    CFG.update(epochs=4, extra_paths=[], run_test=True)

## Visible experiment controls (run one configuration at a time)

The layer count is controlled by `CFG['layers_kept']` below. `None` means use the
backbone preset; for a smaller experiment set it to `12`, then rerun the
training/evaluation cells. Do not change the backbone, data plan, folds, and
seed in the same experiment.

Recommended order: baseline -> layer sweep -> backbone sweep -> seed check ->
private-data ablation -> unseen-set check -> final ensemble.


In [ ]:
# ===================== EXPERIMENT CONTROL PANEL =====================
# Change exactly one field between runs, then rerun from the training cell.
EXPERIMENTS = {
    'baseline_12': dict(backbone='qwen3_4b', layers_kept=12, extra_paths=[]),
    'baseline_24': dict(backbone='qwen3_4b', layers_kept=24, extra_paths=[]),
    'baseline_all': dict(backbone='qwen3_4b', layers_kept=36, extra_paths=[]),
    'backbone_8b_24': dict(backbone='qwen3_8b', layers_kept=24, extra_paths=[]),
    'backbone_14b_27': dict(backbone='qwen3_14b', layers_kept=27, extra_paths=[]),
}
EXPERIMENT_NAME = 'baseline_24'

def select_experiment(name):
    if name not in EXPERIMENTS:
        raise KeyError(f'Unknown experiment {name}; choose from {sorted(EXPERIMENTS)}')
    CFG.update(EXPERIMENTS[name])
    print('selected:', name, '| backbone:', CFG['backbone'], '| layers_kept:', CFG['layers_kept'],
          '| extra_paths:', len(CFG['extra_paths']), '| seeds:', CFG['seeds'])

select_experiment(EXPERIMENT_NAME)
print('Layer sweep to run manually:', [('qwen3_4b', k) for k in (12, 24, 36)])


## 1. Data, folds, overlap checks

In [ ]:
def lab_norm(s): return re.sub(r'[\s_]+', '-', str(s).strip().lower())
def norm_text(t): return re.sub(r'\s+', ' ', str(t)).strip().lower()

if SMOKE:
    rng = np.random.RandomState(0)
    nm = lambda pre, i: pre + chr(97 + i // 26) + chr(97 + i % 26)
    pool = [nm('top', i) for i in range(500)]; fill = [nm('fil', i) for i in range(60)]
    LABELS = [f'model-{k:02d}' for k in range(18)]
    sty = {k: rng.choice(fill, 8, replace=False) for k in range(18)}
    def synth(prompts):
        rows = []
        for p in prompts:
            topic = rng.choice(pool, 14, replace=False)
            for k in range(18):
                w = [rng.choice(topic) if rng.rand() < .45 else rng.choice(sty[k]) if rng.rand() < .35 else rng.choice(fill) for _ in range(70)]
                rows.append((f'{p}_{k}', p, ' '.join(w) + '.', LABELS[k]))
        return pd.DataFrame(rows, columns=['id', 'prompt_id', 'text', 'model'])
    train = synth([f'p{i}' for i in range(30)])
    unseen_raw = synth([f'u{i}' for i in range(6)]); unseen_raw['model_raw'] = unseen_raw['model']
    test = synth([f't{i}' for i in range(8)])[['id', 'text']]
else:
    train = pd.read_csv(find('train.csv')); test = pd.read_csv(find('test.csv'))
    LABELS = sorted(train['model'].unique())
LAB2I = {l: i for i, l in enumerate(LABELS)}; K = len(LABELS)
LABEL_MAP = {lab_norm(l): l for l in LABELS}
LABEL_MAP.update({lab_norm(l).replace('.', '-'): l for l in LABELS})
LABEL_MAP.update({lab_norm(a): b for a, b in LABEL_ALIASES.items()})

def read_any(p):
    if p.endswith('.parquet'): return pd.read_parquet(p)
    if p.endswith('.jsonl'): return pd.read_json(p, lines=True)
    return pd.read_csv(p)

def load_labeled(path):
    df = read_any(path)
    tc = next((c for c in ('text', 'passage', 'generation', 'response') if c in df.columns), None)
    lc = next((c for c in ('model', 'label', 'llm', 'generator') if c in df.columns), None)
    assert tc and lc, f'{path}: need a text column and a model/label column, found {list(df.columns)}'
    out = pd.DataFrame({'text': df[tc].astype(str), 'model_raw': df[lc].astype(str)})
    if 'source_dataset' in df.columns: out['source'] = df['source_dataset'].astype(str)
    if 'prompt_canon' in df.columns:
        pcv = df['prompt_canon'].fillna('').astype(str)
        out['prompt_id'] = pcv.where(pcv.str.startswith('p'), (out['source'].astype(str) if 'source' in out else pd.Series('', index=out.index)) + ':' + pcv)
        out.loc[pcv.eq(''), 'prompt_id'] = np.nan
    for col in ('match_weight', 'panel_labels', 'prompt_evidence', 'flag_kb_proxy', 'n_sources'):
        if col in df.columns: out[col] = df[col].values
    f = lambda s: LABEL_MAP.get(lab_norm(s)) or LABEL_MAP.get(lab_norm(s).replace('.', '-'))
    out['model'] = out.model_raw.map(f)
    if 'prompt_id' not in out: out['prompt_id'] = np.nan
    unk = out[out.model.isna()].model_raw.value_counts()
    if len(unk): print(f'  {os.path.basename(path)}: dropping unknown labels', unk.head(6).to_dict(), '(fix with LABEL_ALIASES)')
    out = out.dropna(subset=['model'])
    return out[out.text.str.len() > 50].reset_index(drop=True)

train_norm = set(train.text.map(norm_text)); test_norm = set(test.text.map(norm_text))
def drop_dupes(df, name):
    n0 = len(df); keep = ~df.text.map(norm_text).isin(train_norm | test_norm)
    df = df[keep].drop_duplicates('text').reset_index(drop=True)
    print(f'  {name}: {n0} -> {len(df)} rows after removing exact duplicates of train/test')
    return df

# optional training-only extras
extra = pd.DataFrame(columns=['text', 'model', 'prompt_id'])
if CFG['extra_paths']:
    frames = [drop_dupes(load_labeled(p), os.path.basename(p)) for p in CFG['extra_paths']]
    extra = pd.concat(frames, ignore_index=True).drop_duplicates('text')
    if len(extra):
        parts = []
        for _, d in extra.groupby('model', sort=False):
            if 'source' in d.columns and CFG['extra_cap_per_source'] < 1:
                per_source = max(1, int(CFG['extra_cap_per_class'] * CFG['extra_cap_per_source']))
                d = pd.concat([s.sample(min(len(s), per_source), random_state=SEED) for _, s in d.groupby('source', sort=False)])
            if CFG['extra_sample_by_match_weight'] and 'match_weight' in d.columns:
                d = d.sample(min(len(d), CFG['extra_cap_per_class']), weights=d['match_weight'].clip(lower=0), random_state=SEED)
            else:
                d = d.sample(min(len(d), CFG['extra_cap_per_class']), random_state=SEED)
            parts.append(d)
        extra = pd.concat(parts, ignore_index=True).reset_index(drop=True)
    else:
        print('  no extra rows survived label matching and de-duplication')
    print('extra rows used for TRAINING only:', len(extra))

# the unseen evaluation set
UN = None
if SMOKE:
    UN = unseen_raw[['text', 'model', 'prompt_id']].copy()
elif CFG['unseen_path']:
    UN = drop_dupes(load_labeled(CFG['unseen_path']), 'unseen')
print('train', len(train), '| prompts', train.prompt_id.nunique(), '| test', len(test), '| unseen', 0 if UN is None else len(UN))

# folds by prompt
prompt_codes = pd.factorize(train.prompt_id)[0]
fold = np.full(len(train), -1)
for f, (_, va) in enumerate(GroupKFold(CFG['n_folds']).split(np.arange(len(train)), groups=prompt_codes)): fold[va] = f
va_mask = fold == CFG['val_fold']
tr_df = pd.concat([train[~va_mask][['text', 'model']].assign(src='orig', w=1.0),
                   extra[['text', 'model']].assign(src='extra', w=CFG['extra_weight'])], ignore_index=True)
va_df = train[va_mask].reset_index(drop=True)
TR_TEXT, TR_Y, TR_W = tr_df.text.tolist(), tr_df.model.map(LAB2I).to_numpy(), tr_df.w.to_numpy(dtype=np.float32)
VA_TEXT, VA_Y = va_df.text.tolist(), va_df.model.map(LAB2I).to_numpy()
UN_TEXT = UN.text.tolist() if UN is not None else []
UN_Y = UN.model.map(LAB2I).to_numpy() if UN is not None else np.array([], int)
TE_TEXT = test.text.tolist()
print(f'train rows {len(TR_TEXT)} (orig {int((tr_df.src=="orig").sum())}, extra {int((tr_df.src=="extra").sum())}) | val rows {len(VA_TEXT)} from {va_df.prompt_id.nunique()} unseen prompts')
print('val prompts disjoint from train prompts:', set(va_df.prompt_id).isdisjoint(set(train[~va_mask].prompt_id)))

def overlap_report(ref_texts, other_texts, name):
    v = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2).fit(list(ref_texts) + list(other_texts))
    A, B = v.transform(ref_texts), v.transform(other_texts)
    mx = np.zeros(B.shape[0])
    for b0 in range(0, B.shape[0], 500): mx[b0:b0 + 500] = (B[b0:b0 + 500] @ A.T).max(axis=1).toarray().ravel()
    print(f'  {name}: nearest-train cosine median {np.median(mx):.2f} | share >0.5: {np.mean(mx > .5):.1%} | >0.7: {np.mean(mx > .7):.1%} | >0.9: {np.mean(mx > .9):.1%}')
    return mx
print('Near-duplicate check against the training rows:')
mx_val = overlap_report(TR_TEXT, VA_TEXT, 'validation fold')
mx_un = overlap_report(TR_TEXT, UN_TEXT, 'unseen set') if UN is not None else None
if mx_un is not None and np.mean(mx_un > .7) > .05:
    print('  WARNING: the unseen set shares many near-duplicates with train; regenerate with new prompts.')

wc = train.text.str.split().str.len()
order = train.assign(wc=wc).groupby('model').wc.median().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.boxplot([train[train.model == m].text.str.split().str.len() for m in order], labels=order, showfliers=False)
ax.set_title('Passage length per model (a length shortcut would show up as separated boxes)'); ax.set_ylabel('words')
plt.setp(ax.get_xticklabels(), rotation=60, ha='right'); plt.tight_layout(); plt.savefig(f'{FIG}/00_lengths.png', dpi=110); plt.show()

## 2. Shared helpers: prompt clustering, Hungarian assignment, metrics

In [ ]:
def content_embed(fit_texts, texts):
    tf = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2, max_df=0.3, token_pattern=r'[A-Za-z]{3,}').fit(fit_texts)
    X = tf.transform(texts); d = max(2, min(128, X.shape[1] - 1))
    E = TruncatedSVD(d, random_state=SEED).fit(tf.transform(fit_texts)).transform(X)
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)

def cluster_prompts(E, k=None, size=18):
    n = len(E); k = max(1, int(round(n / size))) if k is None else max(1, int(k))
    lab = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(E) if 1 < k < n else np.zeros(n, int)
    out, nxt = lab.copy(), lab.max() + 1
    for g in np.unique(lab):
        ii = np.where(lab == g)[0]
        if len(ii) > size:
            kk = int(np.ceil(len(ii) / size)); sub = KMeans(kk, n_init=5, random_state=SEED).fit_predict(E[ii])
            for s in range(1, kk): out[ii[sub == s]] = nxt; nxt += 1
    return out

def assign(logP, groups, bonus):
    out = logP.copy()
    for g in np.unique(groups):
        ii = np.where(groups == g)[0]
        if len(ii) < 2 or len(ii) > logP.shape[1]: continue
        r, c = linear_sum_assignment(-logP[ii]); out[ii[r], c] += bonus
    return out

def metrics(y, P):
    pred = P.argmax(1)
    return dict(acc=accuracy_score(y, pred), bal=balanced_accuracy_score(y, pred),
                loss=log_loss(y, np.clip(P, 1e-7, 1), labels=np.arange(K)) if P.sum(1).min() > 0 else np.nan)

# prompt clusters for the validation rows, computed label-free on ALL train rows (about 150 prompts, like the ~220 in the test)
E_train = content_embed(list(train.text) + TE_TEXT + UN_TEXT, list(train.text))
PG_train = cluster_prompts(E_train)
PG_val = PG_train[va_mask]
TRUE_val = pd.factorize(va_df.prompt_id)[0]
print('cluster ARI vs true prompts (all train rows): %.3f | on the validation rows: %.3f' % (
    adjusted_rand_score(prompt_codes, PG_train), adjusted_rand_score(TRUE_val, PG_val)))
print('cluster size distribution (val rows):', np.bincount(np.bincount(PG_val))[:20].tolist(), '(index = size)')

## 3. Baselines on the same split: TF-IDF logistic regression and a shuffled-label control

In [ ]:
t0 = time.time()
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=150000).fit(TR_TEXT)
vecw = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100000, lowercase=False).fit(TR_TEXT)
from scipy.sparse import hstack
feat = lambda T: hstack([vec.transform(T), vecw.transform(T)]).tocsr()
Xtr_b, Xva_b = feat(TR_TEXT), feat(VA_TEXT)
Xun_b = feat(UN_TEXT) if UN is not None else None
base = LogisticRegression(C=10, max_iter=300).fit(Xtr_b, TR_Y, sample_weight=TR_W)
base_val = base.predict_proba(Xva_b); base_un = base.predict_proba(Xun_b) if UN is not None else None
rs = np.random.RandomState(SEED); ctrl = LogisticRegression(C=10, max_iter=300).fit(Xtr_b, rs.permutation(TR_Y))
ctrl_val = ctrl.predict_proba(Xva_b)
B = dict(base_val=metrics(VA_Y, base_val), ctrl_val=metrics(VA_Y, ctrl_val), base_un=metrics(UN_Y, base_un) if UN is not None else None)
print(f'TF-IDF baseline  val: acc {B["base_val"]["acc"]:.3f}  balanced {B["base_val"]["bal"]:.3f}')
print(f'shuffled-label   val: acc {B["ctrl_val"]["acc"]:.3f}  (chance = {1/K:.3f}; anything clearly higher means the pipeline leaks)')
if UN is not None: print(f'TF-IDF baseline  unseen: acc {B["base_un"]["acc"]:.3f}  balanced {B["base_un"]["bal"]:.3f}')
print(f'baseline time {time.time()-t0:.0f}s')
del Xtr_b; gc.collect()
# provisional submission: a valid file exists early; the final cell overwrites it with the LoRA predictions
if CFG['run_test']:
    prov = pd.DataFrame({'id': test['id'].values, 'model': [LABELS[i] for i in base.predict_proba(feat(TE_TEXT)).argmax(1)]})
    try: prov.to_parquet(f'{OUT}/submission.parquet', index=False); print('provisional submission.parquet written (TF-IDF baseline)')
    except Exception: prov.to_csv(f'{OUT}/submission.csv', index=False); print('provisional submission.csv written (no parquet engine)')

## 4. LoRA fine-tuning
Real mode: truncated Qwen3 decoder + LoRA + mean pooling + linear head, custom loop with gradient accumulation, GradScaler, gradient checkpointing, NaN-step guard, per-epoch validation, train-subset evaluation (memorisation check) and best-epoch restore.

In [ ]:
def find_checkpoint(spec):
    for cfgp in INPUT_FILES:
        if not cfgp.endswith('/config.json'): continue
        d = os.path.dirname(cfgp)
        if d not in ST_DIRS: continue
        try: c = json.load(open(cfgp))
        except Exception: continue
        if (c.get('model_type') == spec['model_type'] and c.get('hidden_size') == spec['hidden'] and c.get('num_hidden_layers') == spec['layers']):
            return d
    return None

STAGED = []
def stage_to_tmp(src, root='/tmp/ntl_stage', workers=4):
    # sequential bulk copy of only the needed files; falls back to the original path if there is no room
    names = [f for f in os.listdir(src) if f.endswith(('.safetensors', '.json', '.txt', '.model', '.jinja'))]
    need = sum(os.path.getsize(os.path.join(src, f)) for f in names)
    os.makedirs(root, exist_ok=True)
    if shutil.disk_usage(root).free < need + 3 * 2**30:
        print(f'  not enough free space in {root} for {need/2**30:.1f} GiB, loading directly from /kaggle/input'); return src
    dst = os.path.join(root, os.path.basename(src.rstrip('/')) + f'_{abs(hash(src)) % 10**6}')
    if os.path.exists(os.path.join(dst, '.done')): return dst
    os.makedirs(dst, exist_ok=True); t = time.time()
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(lambda f: shutil.copyfile(os.path.join(src, f), os.path.join(dst, f)), names))
    open(os.path.join(dst, '.done'), 'w').close(); STAGED.append(dst)
    print(f'  staged {need/2**30:.1f} GiB to {dst} in {time.time()-t:.0f}s ({need/2**20/max(time.time()-t,1e-3):.0f} MiB/s)')
    return dst

def new_hist(): return dict(step=[], loss=[], acc=[], lr=[], epoch=[], epoch_step=[], val_loss=[], val_acc=[], val_bal=[],
                            trainsub_acc=[], trainsub_loss=[], elapsed=[], mem_gb=[], nan_skips=0, stopped_early=False)

def run_training_real():
    import torch.nn as nn, torch.nn.functional as F
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    spec = dict(PRESETS[CFG['backbone']])
    if CFG['layers_kept'] is not None:
        if not (1 <= int(CFG['layers_kept']) <= spec['layers']):
            raise ValueError(f"layers_kept must be between 1 and {spec['layers']} for {CFG['backbone']}")
        spec['keep'] = int(CFG['layers_kept'])
    path = find_checkpoint(spec)
    assert path, f'No {CFG["backbone"]} checkpoint found under /kaggle/input (need model_type={spec["model_type"]}, hidden={spec["hidden"]}, layers={spec["layers"]}). Attach the Qwen3 base model via Add Input > Models.'
    print('backbone path:', path)
    if CFG['stage_to_tmp']: path = stage_to_tmp(path)
    tok = AutoTokenizer.from_pretrained(path)
    eos = tok.eos_token_id; pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos
    def enc(texts):
        ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=CFG['max_len'] - 1)['input_ids']
        return [i + [eos] for i in ids]

    cfg = AutoConfig.from_pretrained(path); cfg.num_hidden_layers = spec['keep']
    if getattr(cfg, 'layer_types', None): cfg.layer_types = cfg.layer_types[:spec['keep']]
    kw = dict(config=cfg, low_cpu_mem_usage=True)
    mode = CFG['gpu_mode']
    if mode == 'auto': mode = 'shard' if (spec['size_gb'] > 10 and N_GPUS >= 2) else 'single'
    if mode == 'single': kw['device_map'] = {'': 0}
    else:
        kw['device_map'] = 'auto'; kw['max_memory'] = {i: ('11GiB' if i == 0 else '13GiB') for i in range(N_GPUS)}
    print('gpu mode:', mode, '| keep layers:', spec['keep'])
    try: base_m = AutoModel.from_pretrained(path, dtype=torch.float16, **kw)
    except TypeError: base_m = AutoModel.from_pretrained(path, torch_dtype=torch.float16, **kw)
    base_m.config.use_cache = False
    base_m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    pm = get_peft_model(base_m, LoraConfig(r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'],
                                           target_modules=CFG['lora_targets'], bias='none'))
    for p in pm.parameters():
        if p.requires_grad: p.data = p.data.float()

    class SeqClf(nn.Module):
        def __init__(self, backbone, hidden, k):
            super().__init__(); self.backbone = backbone; self.head = nn.Linear(hidden, k); self.drop = nn.Dropout(0.1)
        def forward(self, input_ids, attention_mask):
            h = self.backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
            m = attention_mask.to(h.device).unsqueeze(-1).float(); h = h.float()
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
            if self.head.weight.device != pooled.device: self.head.to(pooled.device)
            return self.head(self.drop(pooled))
    clf = SeqClf(pm, cfg.hidden_size, K)
    dev0 = next(pm.parameters()).device
    clf.head.to(f'cuda:{N_GPUS - 1}' if mode == 'shard' else dev0)
    n_train = sum(p.numel() for p in clf.parameters() if p.requires_grad)
    print(f'trainable parameters: {n_train/1e6:.1f}M')

    def collate(rows):
        L = max(len(r) for r in rows)
        x = torch.full((len(rows), L), pad_id, dtype=torch.long); m = torch.zeros((len(rows), L), dtype=torch.long)
        for i, r in enumerate(rows): x[i, :len(r)] = torch.tensor(r); m[i, :len(r)] = 1
        return x, m

    AMP = True
    @torch.no_grad()
    def predict_ids(ids):
        clf.eval(); P = np.zeros((len(ids), K), np.float32); order = np.argsort([len(x) for x in ids])
        for b0 in range(0, len(order), CFG['eval_bs']):
            sel = order[b0:b0 + CFG['eval_bs']]; x, m = collate([ids[i] for i in sel])
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                lg = clf(x.to(dev0), m.to(dev0)).float()
            P[sel] = torch.softmax(lg, -1).cpu().numpy()
        return P
    predict_texts = lambda texts: predict_ids(enc(texts))

    tr_ids, va_ids = enc(TR_TEXT), enc(VA_TEXT)
    rng = np.random.RandomState(SEED)
    orig_pos = np.where((tr_df.src == 'orig').to_numpy())[0]
    sub_pos = rng.choice(orig_pos, min(360, len(orig_pos)), replace=False)

    head_params = list(clf.head.parameters())
    lora_params = [p for n, p in clf.named_parameters() if p.requires_grad and not n.startswith('head.')]
    opt = torch.optim.AdamW([dict(params=lora_params, lr=CFG['lr']), dict(params=head_params, lr=CFG['head_lr'])], weight_decay=CFG['weight_decay'])
    all_params = lora_params + head_params
    try: scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    def batches_for_epoch():
        idx = rng.permutation(len(tr_ids)); chunk = CFG['batch_size'] * 16; out = []
        for c0 in range(0, len(idx), chunk):
            c = idx[c0:c0 + chunk]; c = c[np.argsort([len(tr_ids[i]) for i in c])]
            out += [c[i:i + CFG['batch_size']] for i in range(0, len(c), CFG['batch_size'])]
        return [out[i] for i in rng.permutation(len(out))]
    nb = int(np.ceil(len(tr_ids) / CFG['batch_size'])); steps_per_epoch = int(np.ceil(nb / CFG['grad_accum']))
    total_steps = steps_per_epoch * CFG['epochs']; warm = max(1, int(CFG['warmup_frac'] * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))
    print(f'{len(tr_ids)} train rows | {nb} micro-batches/epoch | {steps_per_epoch} optimizer steps/epoch | {total_steps} total')

    h = new_hist(); t0 = time.time(); gstep = 0; best = (-1, None, None, 0); ema_l, ema_a = None, None
    if len(TR_TEXT) > CFG['max_train_rows_per_epoch']:
        keep = rng.choice(len(TR_TEXT), CFG['max_train_rows_per_epoch'], replace=False)
        TR_TEXT, TR_Y = TR_TEXT[keep], TR_Y[keep]
    for ep in range(1, CFG['epochs'] + 1):
        clf.train(); opt.zero_grad(set_to_none=True); micro = 0; stop = False
        for bi, sel in enumerate(batches_for_epoch()):
            x, m = collate([tr_ids[i] for i in sel]); yb = torch.tensor(TR_Y[sel]); wb = torch.tensor(TR_W[sel])
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                lg = clf(x.to(dev0), m.to(dev0))
            ce = F.cross_entropy(lg.float(), yb.to(lg.device), reduction='none', label_smoothing=CFG['label_smooth'])
            loss = (ce * wb.to(ce.device)).sum() / wb.sum()
            if not torch.isfinite(loss):
                h['nan_skips'] += 1; continue
            scaler.scale(loss / CFG['grad_accum']).backward(); micro += 1
            a = float((lg.argmax(-1).cpu() == yb).float().mean()); l = float(loss)
            ema_l = l if ema_l is None else .9 * ema_l + .1 * l; ema_a = a if ema_a is None else .9 * ema_a + .1 * a
            if micro % CFG['grad_accum'] == 0 or bi == nb - 1:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(all_params, 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                if gstep % CFG['log_every'] == 0 or gstep == 1:
                    h['step'].append(gstep); h['loss'].append(ema_l); h['acc'].append(ema_a); h['lr'].append(sched.get_last_lr()[0])
                if gstep == 10:
                    per = (time.time() - t0) / 10
                    print(f'  speed {per:.1f}s per optimizer step -> ETA for training {per*total_steps/60:.0f} min (plus evaluation)')
                if gstep % 25 == 0:
                    print(f'  epoch {ep} step {gstep}/{total_steps} loss {ema_l:.3f} acc {ema_a:.3f} elapsed {(time.time()-t0)/60:.1f} min', flush=True)
                if (time.time() - t0) / 3600 > CFG['time_budget_hours']:
                    stop = True; h['stopped_early'] = True; print('  time budget reached, stopping this epoch early'); break
        # end of epoch evaluation
        vp = predict_ids(va_ids); mv = metrics(VA_Y, vp)
        sp = predict_ids([tr_ids[i] for i in sub_pos]); ms = metrics(TR_Y[sub_pos], sp)
        h['epoch'].append(ep); h['epoch_step'].append(gstep); h['val_loss'].append(mv['loss']); h['val_acc'].append(mv['acc']); h['val_bal'].append(mv['bal'])
        h['trainsub_acc'].append(ms['acc']); h['trainsub_loss'].append(ms['loss']); h['elapsed'].append((time.time() - t0) / 60)
        h['mem_gb'].append(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPUS)) / 2**30)
        print(f'EPOCH {ep}: val acc {mv["acc"]:.3f} bal {mv["bal"]:.3f} loss {mv["loss"]:.3f} | train-subset acc {ms["acc"]:.3f} | peak GPU mem {h["mem_gb"][-1]:.1f} GiB | {(time.time()-t0)/60:.1f} min', flush=True)
        if mv['bal'] > best[0]:
            best = (mv['bal'], {n: p.detach().cpu().clone() for n, p in clf.named_parameters() if p.requires_grad}, vp, ep)
        if stop: break
    with torch.no_grad():
        for n, p in clf.named_parameters():
            if n in best[1]: p.copy_(best[1][n].to(p.device))
    print(f'restored best epoch {best[3]} (val balanced acc {best[0]:.3f}). NOTE: the epoch was chosen on the validation fold; the unseen set is the clean check.')
    adapter_dir = os.path.join(OUT, f'lora_{CFG["backbone"]}_fold{CFG["val_fold"]}'); os.makedirs(adapter_dir, exist_ok=True)
    torch.save(best[1], os.path.join(adapter_dir, 'trainable_state.pt'))
    R = dict(hist=h, val_probs=best[2], best_epoch=best[3], predict_fn=predict_texts,
             trainsub_y=TR_Y[sub_pos], trainsub_probs=predict_ids([tr_ids[i] for i in sub_pos]))
    R['unseen_probs'] = predict_texts(UN_TEXT) if UN is not None else None
    return R

def run_training_smoke():
    # sklearn stand-in with the same outputs, so every graph and metric cell can be checked without a GPU
    v = TfidfVectorizer(sublinear_tf=True, min_df=2).fit(TR_TEXT); Xt = v.transform(TR_TEXT)
    clf = SGDClassifier(loss='log_loss', alpha=1e-4, random_state=SEED); h = new_hist(); rng = np.random.RandomState(SEED)
    gstep = 0; best = (-1, None, 0); t0 = time.time()
    orig_pos = np.where((tr_df.src == 'orig').to_numpy())[0]; sub_pos = rng.choice(orig_pos, min(360, len(orig_pos)), replace=False)
    for ep in range(1, CFG['epochs'] + 1):
        idx = rng.permutation(len(TR_TEXT))
        for b0 in range(0, len(idx), 32):
            sel = idx[b0:b0 + 32]
            if gstep > 0:
                p = np.clip(clf.predict_proba(Xt[sel]), 1e-7, 1); lo = float(-np.log(p[np.arange(len(sel)), TR_Y[sel]]).mean()); ac = float((p.argmax(1) == TR_Y[sel]).mean())
            else: lo, ac = float(np.log(K)), 1 / K
            clf.partial_fit(Xt[sel], TR_Y[sel], classes=np.arange(K)); gstep += 1
            if gstep % 2 == 0: h['step'].append(gstep); h['loss'].append(lo); h['acc'].append(ac); h['lr'].append(1e-4 * (1 - gstep / 400))
        vp = clf.predict_proba(v.transform(VA_TEXT)); mv = metrics(VA_Y, vp)
        sp = clf.predict_proba(Xt[sub_pos]); ms = metrics(TR_Y[sub_pos], sp)
        h['epoch'].append(ep); h['epoch_step'].append(gstep); h['val_loss'].append(mv['loss']); h['val_acc'].append(mv['acc']); h['val_bal'].append(mv['bal'])
        h['trainsub_acc'].append(ms['acc']); h['trainsub_loss'].append(ms['loss']); h['elapsed'].append((time.time() - t0) / 60); h['mem_gb'].append(0.0)
        print(f'[smoke] EPOCH {ep}: val acc {mv["acc"]:.3f} | train-subset acc {ms["acc"]:.3f}')
        if mv['bal'] > best[0]: best = (mv['bal'], vp, ep)
    pf = lambda texts: clf.predict_proba(v.transform(texts))
    return dict(hist=h, val_probs=best[1], best_epoch=best[2], predict_fn=pf, trainsub_y=TR_Y[sub_pos], trainsub_probs=pf([TR_TEXT[i] for i in sub_pos]),
                unseen_probs=pf(UN_TEXT) if UN is not None else None)

In [ ]:
t_start = time.time()
R = run_training_smoke() if SMOKE else run_training_real()
print(f'\ntraining + evaluation finished in {(time.time()-t_start)/60:.1f} min | best epoch {R["best_epoch"]}')

### Graph 1: training dynamics

In [ ]:
def plot_training(h, base_acc):
    fig, ax = plt.subplots(2, 2, figsize=(13, 8.5))
    s, l, a = np.array(h['step']), np.array(h['loss']), np.array(h['acc'])
    ax[0, 0].plot(s, l, color='tab:blue', label='train loss (EMA)')
    ax[0, 0].plot(h['epoch_step'], h['val_loss'], 'o-', color='tab:red', label='val loss (unseen prompts)')
    ax[0, 0].plot(h['epoch_step'], h['trainsub_loss'], 's--', color='tab:green', label='train-subset loss (eval mode)')
    ax[0, 0].axhline(np.log(K), color='gray', ls=':', label='chance loss'); ax[0, 0].set_title('Loss (train loss includes label smoothing)'); ax[0, 0].set_xlabel('optimizer step'); ax[0, 0].legend()
    ax[0, 1].plot(h['epoch'], h['trainsub_acc'], 's--', color='tab:green', label='train subset')
    ax[0, 1].plot(h['epoch'], h['val_acc'], 'o-', color='tab:red', label='val accuracy')
    ax[0, 1].plot(h['epoch'], h['val_bal'], 'o:', color='tab:orange', label='val balanced acc')
    ax[0, 1].axhline(base_acc, color='tab:purple', ls='--', label=f'TF-IDF baseline val ({base_acc:.2f})'); ax[0, 1].axhline(1 / K, color='gray', ls=':', label='chance')
    ax[0, 1].set_ylim(0, 1.02); ax[0, 1].set_title('Accuracy per epoch'); ax[0, 1].set_xticks(h['epoch']); ax[0, 1].set_xlabel('epoch'); ax[0, 1].legend(fontsize=8)
    ax[1, 0].plot(s, a, color='tab:blue', label='train batch accuracy (EMA)'); ax2 = ax[1, 0].twinx(); ax2.plot(s, h['lr'], color='tab:gray', alpha=.7, label='learning rate')
    ax[1, 0].set_title('Running train accuracy and LR schedule'); ax[1, 0].set_xlabel('optimizer step'); ax[1, 0].set_ylim(0, 1.02)
    gap = np.array(h['trainsub_acc']) - np.array(h['val_acc'])
    ax[1, 1].bar(h['epoch'], gap, color='tab:red', alpha=.7); ax[1, 1].set_title('Memorisation gap = train-subset acc - val acc'); ax[1, 1].set_xlabel('epoch'); ax[1, 1].set_xticks(h['epoch'])
    for e, g in zip(h['epoch'], gap): ax[1, 1].text(e, g, f'{g:.2f}', ha='center', va='bottom')
    plt.tight_layout(); plt.savefig(f'{FIG}/01_training_dynamics.png', dpi=110); plt.show()
plot_training(R['hist'], B['base_val']['acc'])
print('NaN/inf steps skipped:', R['hist']['nan_skips'], '| stopped early on time budget:', R['hist']['stopped_early'])

## 5. Evaluation on unseen prompts: raw, with group assignment, and on your generated set

In [ ]:
RES = {}
vp = R['val_probs']; logv = np.log(np.clip(vp, 1e-9, 1))
RES['TF-IDF baseline (val)'] = B['base_val']; RES['Shuffled-label control (val)'] = B['ctrl_val']; RES['LoRA raw (val)'] = metrics(VA_Y, vp)
sweep = {}
for b in (0, 0.5, 1, 2, 4, 1e3):
    sweep[b] = balanced_accuracy_score(VA_Y, assign(logv, PG_val, b).argmax(1))
BEST_BONUS = max(sweep, key=sweep.get)
RES['LoRA + Hungarian, pseudo-groups (val)'] = metrics(VA_Y, np.exp(assign(logv, PG_val, BEST_BONUS)))
RES['LoRA + Hungarian, TRUE groups (val, oracle)'] = metrics(VA_Y, np.exp(assign(logv, TRUE_val, 1e3)))
RES['TF-IDF + Hungarian, pseudo-groups (val)'] = metrics(VA_Y, np.exp(assign(np.log(np.clip(base_val, 1e-9, 1)), PG_val, BEST_BONUS)))
if UN is not None and R['unseen_probs'] is not None:
    up = R['unseen_probs']; logu = np.log(np.clip(up, 1e-9, 1))
    if UN.prompt_id.notna().any():
        true_un = pd.factorize(UN.prompt_id)[0]; k_un = len(np.unique(true_un))
    else: true_un, k_un = None, None
    E_un = content_embed(list(train.text) + TE_TEXT + UN_TEXT, UN_TEXT)
    PG_un = cluster_prompts(E_un, k=k_un)
    RES['TF-IDF baseline (UNSEEN set)'] = B['base_un']; RES['LoRA raw (UNSEEN set)'] = metrics(UN_Y, up)
    RES['LoRA + Hungarian, pseudo-groups (UNSEEN set)'] = metrics(UN_Y, np.exp(assign(logu, PG_un, BEST_BONUS)))
    if true_un is not None:
        RES['LoRA + Hungarian, TRUE groups (UNSEEN set, oracle)'] = metrics(UN_Y, np.exp(assign(logu, true_un, 1e3)))
        print('unseen cluster ARI: %.3f' % adjusted_rand_score(true_un, PG_un))
sub_m = metrics(R['trainsub_y'], R['trainsub_probs']); RES['LoRA on train subset (in-sample, memorisation)'] = sub_m
summary = pd.DataFrame(RES).T[['acc', 'bal', 'loss']].round(4)
display(summary); summary.to_csv(f'{OUT}/results_summary.csv')
print('chance accuracy:', round(1 / K, 4), '| Hungarian bonus chosen on val:', BEST_BONUS)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw=dict(width_ratios=[1.5, 1]))
names = list(summary.index); vals = summary['bal'].values
colors = ['tab:gray' if 'Shuffled' in n else 'tab:purple' if 'TF-IDF' in n else 'tab:green' if 'in-sample' in n else 'tab:red' if 'UNSEEN' in n else 'tab:blue' for n in names]
ax[0].barh(range(len(names)), vals, color=colors); ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels(names, fontsize=8); ax[0].invert_yaxis()
ax[0].axvline(1 / K, color='gray', ls=':'); ax[0].set_xlim(0, 1.05); ax[0].set_xlabel('balanced accuracy')
for i, v in enumerate(vals): ax[0].text(v + .005, i, f'{v:.3f}', va='center', fontsize=8)
ax[0].set_title('Is it working? (red = your unseen set, green = in-sample, so ignore for quality)')
ax[1].plot(list(range(len(sweep))), list(sweep.values()), 'o-'); ax[1].set_xticks(range(len(sweep))); ax[1].set_xticklabels([str(k) for k in sweep])
ax[1].set_xlabel('Hungarian bonus (val, pseudo-groups)'); ax[1].set_ylabel('balanced accuracy'); ax[1].set_title('How hard to push the one-per-prompt rule')
plt.tight_layout(); plt.savefig(f'{FIG}/02_does_it_work.png', dpi=110); plt.show()

In [ ]:
cm = confusion_matrix(VA_Y, vp.argmax(1), labels=np.arange(K)); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw=dict(width_ratios=[1.15, 1]))
im = ax[0].imshow(cmn, cmap='Blues', vmin=0, vmax=1); ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K))
ax[0].set_xticklabels(LABELS, rotation=75, ha='right', fontsize=7); ax[0].set_yticklabels(LABELS, fontsize=7)
ax[0].set_title('Validation confusion matrix (row-normalised)'); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('true'); plt.colorbar(im, ax=ax[0], fraction=.046)
rec = np.diag(cmn); w = .4; x = np.arange(K)
ax[1].bar(x - w / 2, rec, w, label='LoRA val recall', color='tab:blue')
base_rec = np.diag(confusion_matrix(VA_Y, base_val.argmax(1), labels=np.arange(K))) / np.maximum(cm.sum(1), 1)
ax[1].bar(x + w / 2, base_rec, w, label='TF-IDF val recall', color='tab:purple', alpha=.7)
ax[1].set_xticks(x); ax[1].set_xticklabels(LABELS, rotation=75, ha='right', fontsize=7); ax[1].set_ylim(0, 1.05); ax[1].legend(); ax[1].set_title('Per-class recall')
plt.tight_layout(); plt.savefig(f'{FIG}/03_confusions.png', dpi=110); plt.show()
worst = np.argsort(rec)[:3]; print('weakest classes on val:', [(LABELS[i], round(float(rec[i]), 2)) for i in worst])

## 6. Verdict (read this before spending more compute)

In [ ]:
lv = RES['LoRA raw (val)']['bal']; bv = RES['TF-IDF baseline (val)']['bal']; cv = RES['Shuffled-label control (val)']['acc']
h = R['hist']; gap = h['trainsub_acc'][-1] - h['val_acc'][-1]
print(f'1. Shuffled-label control accuracy {cv:.3f} vs chance {1/K:.3f}:', 'OK, no leakage signal' if cv < 2 / K else 'SUSPICIOUS: check the split and the data for leakage')
print(f'2. LoRA beats TF-IDF by {100*(lv-bv):+.1f} points of balanced accuracy on unseen val prompts:', 'fine-tuning is adding something' if lv - bv > .03 else 'no clear gain yet, do not scale up (more epochs, more data or a bigger backbone first)')
print(f'3. Memorisation gap at the last epoch (train-subset minus val): {gap:.2f}', '(large, expected after several epochs; judge by val, not by this)' if gap > .3 else '')
if len(h['val_bal']) >= 2:
    d = h['val_bal'][-1] - h['val_bal'][-2]
    print(f'4. Last epoch changed val balanced acc by {100*d:+.1f} points:', 'still improving, train longer' if d > .01 else 'plateau or overfitting, keep the best epoch')
g = RES['LoRA + Hungarian, pseudo-groups (val)']['bal'] - lv
print(f'5. Hungarian with pseudo-groups changes val balanced acc by {100*g:+.1f} points (oracle groups: {100*(RES["LoRA + Hungarian, TRUE groups (val, oracle)"]["bal"]-lv):+.1f})')
if UN is not None and 'LoRA raw (UNSEEN set)' in RES:
    du = RES['LoRA raw (UNSEEN set)']['bal'] - lv
    print(f'6. Your unseen set vs the validation fold: {100*du:+.1f} points.', 'big drop: either distribution shift in how you generated it, or the val fold flatters the model' if du < -.10 else 'consistent')
    print(f'   LoRA vs TF-IDF on the unseen set: {100*(RES["LoRA raw (UNSEEN set)"]["bal"]-RES["TF-IDF baseline (UNSEEN set)"]["bal"]):+.1f} points')
else:
    print('6. No unseen set provided yet: set CFG["unseen_path"] to score your generated data.')

## 7. Test predictions and saved artifacts

In [ ]:
json.dump(R['hist'], open(f'{OUT}/history.json', 'w'))
np.save(f'{OUT}/val_probs.npy', R['val_probs'])
if R['unseen_probs'] is not None: np.save(f'{OUT}/unseen_probs.npy', R['unseen_probs'])
if CFG['run_test']:
    t0 = time.time()
    tp = R['predict_fn'](TE_TEXT); np.save(f'{OUT}/test_probs.npy', tp)
    if len(TE_TEXT) >= 36:
        E_te = content_embed(list(train.text) + TE_TEXT + UN_TEXT, TE_TEXT); TEST_G = cluster_prompts(E_te)
        final = assign(np.log(np.clip(tp, 1e-9, 1)), TEST_G, BEST_BONUS)
        print('test prompt groups:', len(np.unique(TEST_G)), '| rows changed by assignment:', int((final.argmax(1) != tp.argmax(1)).sum()))
    else:
        final = np.log(np.clip(tp, 1e-9, 1))
    sub = pd.DataFrame({'id': test['id'].values, 'model': [LABELS[i] for i in final.argmax(1)]})
    assert sub.id.is_unique and len(sub) == len(test)
    try: sub.to_parquet(f'{OUT}/submission.parquet', index=False); print('wrote submission.parquet')
    except Exception as e: sub.to_csv(f'{OUT}/submission.csv', index=False); print('parquet unavailable, wrote submission.csv', e)
    print(sub.model.value_counts().head(18).to_string()); print(f'test prediction {time.time()-t0:.0f}s')
for d in STAGED: shutil.rmtree(d, ignore_errors=True)
print('saved:', sorted(os.listdir(OUT))[:30])

## How to generate the unseen set (so the result means something)
- **Prompts must be new.** Write prompts that are not in `train.csv` (the overlap check above flags near-duplicates). New topics matter more than new wording.
- **Columns:** `text`, `model` (the exact competition label spelling, or add aliases to `LABEL_ALIASES`), and ideally `prompt_id`. With `prompt_id` the notebook can also score the oracle-group assignment.
- **Coverage:** with credits you probably cannot reach all 18 models. That is fine: metrics are computed on the classes present, but the 18-way output can still confuse an absent class, so interpret absolute numbers with care.
- **Match the competition format:** self-contained paragraphs, similar length range, no system-prompt tricks, default sampling settings. Light cleanup only.
- **It is a stress test, not a leaderboard predictor.** Different API settings change style. A large drop on the unseen set can mean shift in how it was generated, not that the model memorised.
- Keep it **evaluation-only**. Once you train on it, you need a fresh set.

## Next steps
1. Read the verdict. If LoRA beats TF-IDF on unseen prompts, change one thing at a time: `epochs`, then `backbone='qwen3_8b'`, then `extra_paths` (training-only extras).
2. Run `val_fold` 1 to 4 to get a stable estimate (each run saves its own adapter folder).
3. Use `val_probs.npy` and `test_probs.npy` from several runs in a stacker; only fold-out probabilities are honest inputs for it.

In [ ]:
print('balanced-accuracy seeds:', CFG['seeds'], '| per-epoch cap:', CFG['max_train_rows_per_epoch'])
